# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [37]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [38]:
import os
import chromadb
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool


In [39]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [40]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most relevant games in the vector DB
    args:
    - query: a question about game industry.
    
    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    
    metadatas = results.get('metadatas', [[]])[0] if results else []
    
    return [
        {
            "Platform": m.get("Platform"),
            "Name": m.get("Name"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Description": m.get("Description")
        }
        for m in metadatas
    ]

#### Evaluate Retrieval Tool

In [41]:
import ast
import json as _json
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


def _parse_docs(retrieved_docs_str: str) -> list:
    """
    Parse the retrieved_docs argument, which the LLM sends as a repr/JSON string
    of the list returned by retrieve_game.
    """
    if isinstance(retrieved_docs_str, list):
        return retrieved_docs_str  # already a proper list
    for parser in (_json.loads, ast.literal_eval):
        try:
            result = parser(retrieved_docs_str)
            if isinstance(result, list):
                return result
            if isinstance(result, dict):
                return [result]
        except Exception:
            continue
    return []


@tool
def evaluate_retrieval(question: str, retrieved_docs: str):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question.
    args: 
    - question: original question from user
    - retrieved_docs: JSON or repr string of the list of retrieved documents from retrieve_game
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    docs = _parse_docs(retrieved_docs)

    docs_text = "\n".join([
        f"- Name: {d.get('Name', 'Unknown')} | Platform: {d.get('Platform', 'Unknown')} | Release Year: {d.get('YearOfRelease', 'Unknown')} | Description: {d.get('Description', '')}"
        for d in docs
    ])

    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Query: {question}

Retrieved Documents:
{docs_text}"""

    llm = LLM(api_key=OPENAI_API_KEY)
    response = llm.invoke(prompt, response_format=EvaluationReport)

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [42]:
from tavily import TavilyClient


@tool
def game_web_search(question: str):
    """
    Web search: Finds relevant results on the web about the game industry.
    Use this when the vector DB does not have enough information to answer the question.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=5)

    return [
        {
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        }
        for result in response.get("results", [])
    ]

### Agent

In [43]:
SYSTEM_INSTRUCTIONS = """You are UdaPlay, an expert AI research agent for the video game industry.

Your goal is to answer questions accurately about video games using the following workflow:
1. Use `retrieve_game` to search the internal knowledge base first.
2. Use `evaluate_retrieval` to assess whether the retrieved documents are sufficient.
   - Pass the original question and the full result from `retrieve_game` as a string.
   - `evaluate_retrieval` returns an EvaluationReport with two fields:
       * useful (bool): true if the documents contain enough information to answer the question.
       * description (str): explanation of why the documents are or are not sufficient.
   - If useful is true, proceed directly to synthesizing an answer from the retrieved documents.
   - If useful is false, you MUST call `game_web_search` before answering.
3. Use `game_web_search` only when evaluate_retrieval returns useful=false.
4. Synthesize all available information into a clear, accurate, and concise answer.

Guidelines:
- Always try the internal knowledge base before searching the web.
- Trust the EvaluationReport: if useful=true, do NOT call game_web_search.
- Be factual and cite sources when using web results.
- If no information is found anywhere, say so clearly.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [44]:
import ast as _ast_trace
import json as _json_trace


def _parse_eval_msg(content: str) -> dict:
    """Unwrap the double-serialised evaluation ToolMessage content."""
    try:
        payload = _json_trace.loads(content)
        if isinstance(payload, str):
            try:
                return _json_trace.loads(payload)
            except Exception:
                return _ast_trace.literal_eval(payload)
        return payload
    except Exception:
        return {}


def run_structured(question: str, session_id: str):
    """Run a query and print a structured step-by-step report."""
    print(f"\nQuery: {question}")
    print("=" * 70)

    run = agent.invoke(question, session_id=session_id)
    state = run.get_final_state()

    step = 1
    retrieved_docs_text = None
    eval_useful = None
    web_results = []
    tools_called = []

    for msg in state["messages"]:
        # Retrieval step
        if isinstance(msg, ToolMessage) and msg.name == "retrieve_game":
            try:
                raw = _json_trace.loads(msg.content)
                docs = _ast_trace.literal_eval(raw) if isinstance(raw, str) else raw
                if isinstance(docs, list) and docs:
                    retrieved_docs_text = "\n".join(
                        f"  • {d.get('Name','?')} ({d.get('Platform','?')}, {d.get('YearOfRelease','?')}): {d.get('Description','')}"
                        for d in docs[:3]
                    )
            except Exception:
                retrieved_docs_text = msg.content[:200]
            print(f"\nStep {step} – Retrieval (retrieve_game):")
            print(f"  Retrieved from vector DB:")
            print(retrieved_docs_text or "  (no results)")
            step += 1

        # Evaluation step
        if isinstance(msg, ToolMessage) and msg.name == "evaluate_retrieval":
            ev = _parse_eval_msg(msg.content)
            eval_useful = ev.get("useful", "?")
            desc = str(ev.get("description", ""))[:160]
            print(f"\nStep {step} – Evaluation (evaluate_retrieval):")
            print(f"  Result: useful = {eval_useful}")
            print(f"  Reason: {desc}...")
            step += 1

        # Web search step
        if isinstance(msg, ToolMessage) and msg.name == "game_web_search":
            try:
                raw = _json_trace.loads(msg.content)
                results = _ast_trace.literal_eval(raw) if isinstance(raw, str) else raw
                web_results = results if isinstance(results, list) else []
            except Exception:
                web_results = []
            print(f"\nStep {step} – Web Search (game_web_search):")
            if web_results:
                print(f"  Top result: {web_results[0].get('title','?')} — {web_results[0].get('url','?')}")
            else:
                print("  (no web results returned)")
            step += 1

        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                tools_called.append(call.function.name)

    # ── Final answer
    answer = next(
        (m.content for m in reversed(state["messages"])
         if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )
    source = "Web" if web_results else "Local Database"
    citation = f"(Source: {source})"
    if web_results:
        citation = f"(Source: Web — {web_results[0].get('url', 'unknown')})"

    print(f"\nStep {step} – Final Answer:")
    print(f"  {answer}")
    print(f"  {citation}")
    print()

    return {
        "question": question,
        "tools": tools_called,
        "eval_useful": eval_useful,
        "web_fallback": bool(web_results),
        "answer": answer,
    }


# Each query gets its own session_id so message histories don't bleed across runs
queries = [
    ("When was Pokémon Gold and Silver released?",     "session_q1"),
    ("Was Mortal Kombat X released for PlayStation 5?", "session_q2"),
    ("Which was the first 3D platformer Mario game?",   "session_q3"),
]

trace_results = [run_structured(q, sid) for q, sid in queries]



Query: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Step 1 – Retrieval (retrieve_game):
  Retrieved from vector DB:
  • Pokémon Gold and Silver (Game Boy Color, 1999): Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
  • Pokémon Ruby and Sapphire (Game Boy Advance, 2002): Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
  • Mario Kart 8 Deluxe (Nintendo Switch, 2017): An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.

Step 2 – Evaluation (evaluate_retrieval):
  Result: useful = True
  Reason: The retrie

### Performance Summary

Each query above prints a structured step-by-step trace showing retrieval, evaluation, optional web search, and a cited final answer.

| # | Query | Retrieval | Evaluation | Web Fallback | Source |
|---|-------|-----------|------------|--------------|--------|
| 1 | When was Pokémon Gold and Silver released? | Relevant results found in local DB | useful = True | No | Local Database |
| 2 | Was Mortal Kombat X released for PlayStation 5? | Results retrieved but not relevant to query | useful = False | **Yes** | Web |
| 3 | Which was the first 3D platformer Mario game? | Relevant results found in local DB | useful = True | No | Local Database |

**Observations:**
- **Query 1 & 3** — the vector DB returned game records directly relevant to the question. `evaluate_retrieval` returned `useful=True` and the agent answered from local data, cited as `(Source: Local Database)`.
- **Query 2** — the vector DB returned results (e.g. Spider-Man 2, Halo Infinite), but none were about Mortal Kombat X on PS5. `evaluate_retrieval` correctly returned `useful=False`, triggering `game_web_search`. The final answer is cited with the web URL.
- The workflow (retrieve → evaluate → optional web search → synthesise) behaves as designed: web search is only triggered when retrieved documents are genuinely insufficient, not as a default fallback.

### (Optional) Advanced

In [45]:
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.agents import AgentState

import json


# ── Long-term memory setup ──────────────────────────────────────────────────

vector_store_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=vector_store_manager)

USER_ID = "udaplay_user"


@tool
def save_to_memory(fact: str):
    """
    Save an important fact or insight to long-term memory for future reference.
    args:
    - fact: a fact or insight worth remembering about the game industry.
    """
    fragment = MemoryFragment(content=fact, owner=USER_ID, namespace="game_facts")
    long_term_memory.register(fragment)
    return {"saved": True, "content": fact}


@tool
def recall_from_memory(query: str):
    """
    Search long-term memory for previously stored facts relevant to the query.
    args:
    - query: a question or topic to search for in long-term memory.
    """
    result = long_term_memory.search(query_text=query, owner=USER_ID, namespace="game_facts", limit=3)
    return [fragment.content for fragment in result.fragments]


def make_tool_node(tool_fn):
    """Wrap a tool into a state machine Step that runs all pending calls for it."""
    def step_logic(state: AgentState) -> AgentState:
        tool_calls = state.get("current_tool_calls") or []
        tool_messages = []

        for call in tool_calls:
            if call.function.name != tool_fn.name:
                continue
            args = json.loads(call.function.arguments)
            result = str(tool_fn(**args))
            tool_messages.append(
                ToolMessage(content=json.dumps(result), tool_call_id=call.id, name=tool_fn.name)
            )

        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"],
        }

    return Step[AgentState](tool_fn.name, step_logic)


extended_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS + "\nYou also have access to `save_to_memory` and `recall_from_memory` tools. "
                 "Use `recall_from_memory` at the start of each query, and `save_to_memory` when you learn a new fact.",
    tools=[retrieve_game, evaluate_retrieval, game_web_search, save_to_memory, recall_from_memory],
    temperature=0.3,
)

# Verify the state machine nodes are wired correctly
print("Agent state machine steps:", list(extended_agent.workflow.steps.keys()))
print("Long-term memory ready for user:", USER_ID)

Agent state machine steps: ['__entry__', 'message_prep', 'llm_processor', 'tool_executor', '__termination__']
Long-term memory ready for user: udaplay_user


### Multi-Turn Conversation (Stateful)

The agent maintains conversation history per `session_id`. When the same `session_id` is reused, the previous messages are restored from `ShortTermMemory` and passed to the next invocation — enabling context-aware follow-up questions.

In [49]:
def get_answer(run) -> str:
    """Extract the last non-empty AI response from a run."""
    state = run.get_final_state()
    return next(
        (m.content for m in reversed(state["messages"]) if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )


def show_context_summary(agent_obj, session_id: str, label: str):
    """Print a short summary of what messages are currently stored in the session."""
    try:
        last_run = agent_obj.memory.get_last_object(session_id)
        if last_run:
            msgs = last_run.get_final_state().get("messages", [])
            user_msgs = [m.content for m in msgs if isinstance(m, UserMessage) and m.content]
            ai_msgs  = [m.content[:80] for m in msgs if isinstance(m, AIMessage) and m.content]
            print(f"  [{label}] Session '{session_id}' carries {len(msgs)} messages")
            print(f"  User turns in history : {user_msgs}")
            print(f"  Last AI snippet       : {ai_msgs[-1] + '...' if ai_msgs else '(none)'}")
        else:
            print(f"  [{label}] Session '{session_id}' is empty (first turn)")
    except Exception as e:
        print(f"  [{label}] Could not inspect session: {e}")

MULTI_SESSION = "witcher_session"

# Turn 1: Establish context
print("=" * 45)
print("Turn 1 | Q: Tell me about The Witcher 3")
print("-" * 70)
show_context_summary(agent, MULTI_SESSION, "context before Turn 1")

run1 = agent.invoke("Tell me about The Witcher 3", session_id=MULTI_SESSION)
answer1 = get_answer(run1)
print(f"A: {answer1}")

# Turn 2: Ambiguous follow-up – must resolve "it" from history
print("Turn 2 | Q: What genre is it?")
print("-" * 45)
show_context_summary(agent, MULTI_SESSION, "context before Turn 2")

run2 = agent.invoke("What genre is it?", session_id=MULTI_SESSION)
answer2 = get_answer(run2)
print(f"A: {answer2}")

# Verify context was actually used and that it still mentions The Witcher 3
resolved = "witcher" in answer2.lower()
print(f"[Context check] Answer mentions 'The Witcher 3': {resolved}")
if not resolved:
    print("  WARNING: Turn 2 may not have used prior context correctly.")

# Turn 3: Trying another ambiguous follow-up 
print("=" * 70)
print("Turn 3 | Q: Who developed it?")
print("-" * 70)
show_context_summary(agent, MULTI_SESSION, "context before Turn 3")

run3 = agent.invoke("Who developed it?", session_id=MULTI_SESSION)
answer3 = get_answer(run3)
print(f"A: {answer3}")

resolved3 = "witcher" in answer3.lower() or "cd projekt" in answer3.lower()
print(f"[Context check] Answer references The Witcher 3 / CD Projekt: {resolved3}")

# ── Session growth proof ────────────────────────────────────────────────────
final_msgs = run3.get_final_state().get("messages", [])
print(f"{'=' * 70}")
print(f"Total messages in session after 3 turns: {len(final_msgs)}")
print("(System prompt + 3 user turns + all assistant replies + all tool messages)")
print("This confirms the agent accumulates and reuses conversation history.")

Turn 1 | Q: Tell me about The Witcher 3
----------------------------------------------------------------------
  [context before Turn 1] Session 'witcher_session' carries 25 messages
  User turns in history : ['Tell me about The Witcher 3', 'What genre is it?', 'Who developed it?']
  Last AI snippet       : *The Witcher 3: Wild Hunt* was developed by **CD Projekt Red**, a Polish video g...
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: **The Witcher 3: Wild Hunt** is an action role-playing game developed by CD Projekt Red, released in 2015. It is the third installment in *The Witcher* series, which is based on the book series by Polish author Andrzej Sapkowski. The game is set in a vast open world and follows the story of Geralt of Rivia, a monster hunter known as a Witcher, as he searches for his adopted daughter, Ciri, who is being pursued by the Wild Hunt, a gr